시내버스수 / 면적, 시외버스 수 / 면적, 기차역 수 / 면적

In [ ]:
import pandas as pd
bus_in = pd.read_csv(r"C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대중교통 위치데이터(2023)\전국 버스정류장 위치정보 (2023).csv",encoding = "euc-kr")
bus_out = pd.read_csv(r"C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대중교통 위치데이터(2023)\시외고속버스터미널 (버스타고).csv")
train = pd.read_csv(r"C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대중교통 위치데이터(2023)\전국 기차역 데이터 (크롤링).csv")
land = pd.read_csv(r"C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\국토 면적\시도별 토지면적.csv")



In [ ]:
print(bus_in.head())
print("-"*50)
print(bus_out.head())
print("-"*50)
print(train.head())
print("-"*50)
print(land.head())

In [ ]:
print(bus_in["도시명"].unique())
print(bus_out["지역"].unique())
print(train["지역"].unique())


In [ ]:
bus_in['지역'] = bus_in['도시명'].apply(lambda x: x.split()[0])

In [ ]:
print(bus_in["지역"].unique())
print(bus_out["지역"].unique())
print(train["지역"].unique())
print(land["지역"].unique())

In [ ]:
region_mapping = {
    '서울특별시': '서울특별시',
    '부산광역시': '부산광역시',
    '대구광역시': '대구광역시',
    '인천광역시': '인천광역시',
    '광주광역시': '광주광역시',
    '대전광역시': '대전광역시',
    '울산광역시': '울산광역시',
    '세종특별자치시': '세종특별자치시',
    '경기도': '경기도',
    '강원도': '강원도',
    '충청북도': '충청북도',
    '충청남도': '충청남도',
    '전라북도': '전라북도',
    '전라남도': '전라남도',
    '경상북도': '경상북도',
    '경상남도': '경상남도',
    '제주특별자치도': '제주특별자치도',
    '서울': '서울특별시',
    '부산': '부산광역시',
    '대구': '대구광역시',
    '인천': '인천광역시',
    '광주': '광주광역시',
    '대전': '대전광역시',
    '울산': '울산광역시',
    '세종시': '세종특별자치시',
    '강원특별자치도': '강원도',
    '경상북도': '경상북도',
    '경상남도': '경상남도',
    '충청북도': '충청북도',
    '충청남도': '충청남도',
    '전라북도': '전라북도',
    '전라남도': '전라남도',
    '제주': '제주특별자치도',
}

# 각 데이터프레임에서 '지역' 컬럼 통일
bus_in['지역'] = bus_in['도시명'].apply(lambda x: region_mapping.get(x.split()[0], x.split()[0]))
bus_out['지역'] = bus_out['지역'].map(region_mapping)
train['지역'] = train['지역'].map(region_mapping)
land['지역'] = land['지역'].map(region_mapping)

In [ ]:
# 시내버스 데이터 처리
bus_in_counts = bus_in['지역'].value_counts().reset_index()
bus_in_counts.columns = ['지역', '시내버스수']

# 시외버스 데이터 처리
bus_out_counts = bus_out['지역'].value_counts().reset_index()
bus_out_counts.columns = ['지역', '시외버스수']

# 기차역 데이터 처리
train_counts = train['지역'].value_counts().reset_index()
train_counts.columns = ['지역', '기차역수']

# 면적 데이터 처리
land.columns = ['지역', '면적']  # 컬럼명 변경

# 데이터 병합
combined = land.merge(bus_in_counts, on='지역', how='left')
combined = combined.merge(bus_out_counts, on='지역', how='left')
combined = combined.merge(train_counts, on='지역', how='left')

# NaN 값을 0으로 대체 (없는 경우를 위한 처리)
combined = combined.fillna(0)

# 면적 열을 숫자로 변환
combined['면적'] = combined['면적'].replace({',': ''}, regex=True).astype(float)

# 비율 계산
combined['시내버스수/면적'] = combined['시내버스수'] / combined['면적']
combined['시외버스수/면적'] = combined['시외버스수'] / combined['면적']
combined['기차역수/면적'] = combined['기차역수'] / combined['면적']

# 결과 출력
print(combined[['지역', '시내버스수/면적', '시외버스수/면적', '기차역수/면적']])

df = pd.DataFrame(combined[['지역', '시내버스수/면적', '시외버스수/면적', '기차역수/면적']])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import koreanize_matplotlib
import warnings

# 경고 무시 설정
warnings.filterwarnings("ignore")

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우에서는 'Malgun Gothic'을 사용합니다.
# matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS에서는 'AppleGothic'을 사용합니다.
# matplotlib.rcParams['font.family'] = 'NanumGothic'  # Ubuntu와 같은 리눅스에서는 'NanumGothic'을 사용합니다.
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 시내버스 비율 그래프 (내림차순 정렬)
df_sorted_bus = df.sort_values(by='시내버스수/면적', ascending=False)
plt.figure(figsize=(14, 6))
bars = plt.bar(df_sorted_bus['지역'], df_sorted_bus['시내버스수/면적'], color='blue')
average_bus = df_sorted_bus['시내버스수/면적'].mean()
plt.axhline(y=average_bus, color='red', linestyle='--', label=f'평균: {average_bus:.6f}')
plt.text(x=len(df_sorted_bus['지역'])-1, y=average_bus, s=f'평균: {average_bus:.6f}', color='red', 
         ha='right', va='bottom', fontsize=12)
plt.xticks(rotation=90)
plt.xlabel('지역')
plt.ylabel('시내버스수/면적')
plt.title('시내버스수/면적 by 지역')
plt.legend()
plt.tight_layout()
plt.show()

# 시외버스 비율 그래프 (내림차순 정렬)
df_sorted_express = df.sort_values(by='시외버스수/면적', ascending=False)
plt.figure(figsize=(14, 6))
bars = plt.bar(df_sorted_express['지역'], df_sorted_express['시외버스수/면적'], color='green')
average_express = df_sorted_express['시외버스수/면적'].mean()
plt.axhline(y=average_express, color='red', linestyle='--', label=f'평균: {average_express:.6f}')
plt.text(x=len(df_sorted_express['지역'])-1, y=average_express, s=f'평균: {average_express:.6f}', color='red', 
         ha='right', va='bottom', fontsize=12)
plt.xticks(rotation=90)
plt.xlabel('지역')
plt.ylabel('시외버스수/면적')
plt.title('시외버스수/면적 by 지역')
plt.legend()
plt.tight_layout()
plt.show()

# 기차역 비율 그래프 (내림차순 정렬)
df_sorted_train = df.sort_values(by='기차역수/면적', ascending=False)
plt.figure(figsize=(14, 6))
bars = plt.bar(df_sorted_train['지역'], df_sorted_train['기차역수/면적'], color='red')
average_train = df_sorted_train['기차역수/면적'].mean()
plt.axhline(y=average_train, color='red', linestyle='--', label=f'평균: {average_train:.6f}')
plt.text(x=len(df_sorted_train['지역'])-1, y=average_train, s=f'평균: {average_train:.6f}', color='red', 
         ha='right', va='bottom', fontsize=12)
plt.xticks(rotation=90)
plt.xlabel('지역')
plt.ylabel('기차역수/면적')
plt.title('기차역수/면적 by 지역')
plt.legend()
plt.tight_layout()
plt.show()